<img src="images/logo/selene-logo-640.png" style="max-height:75px;" alt="SELENE Logo" />

**Disclaimer:** This Jupyter Notebook contains content generated with the assistance of AI. While every effort has been made to review and validate the outputs, users should independently verify critical information before relying on it. The SELENE notebook repository is constantly evolving. We recommend downloading or pulling the latest version of this notebook from Github.

# MLP-based Language Models

This notebook introduces a simple **multilayer perceptron (MLP)** as a first step toward understanding neural language models. Before diving into complex architectures, it is useful to see how far we can get with a straightforward feed-forward network: embeddings, a few fully connected layers, and a softmax output. Despite its simplicity, this setup already captures the core idea behind language modeling—learning to predict the next word given a sequence of previous words. By stripping away architectural complexity, we can focus on the mechanics of data preparation, training objectives, and how neural networks learn distributions over text.

The central training objective in this notebook follows the standard formulation of language modeling: given a context of previous tokens, the model predicts the most likely next token. In practice, this means constructing fixed-size input windows and training the network to maximize the probability of the correct next word. Even though the model is not sequential in nature, it can still approximate conditional probabilities over short contexts. This highlights an important insight: language modeling is less about the specific architecture and more about the **learning objective and training setup**.

That said, MLPs come with clear limitations. Because they operate on fixed-size inputs, they lack any built-in mechanism to capture **long-term dependencies** in text. Information outside the chosen context window is simply inaccessible to the model, no matter how deep or wide the network becomes. This makes MLPs fundamentally ill-suited for realistic language modeling tasks, where meaning often depends on relationships spanning entire sentences or even paragraphs. Architectures such as recurrent neural networks and transformers were developed precisely to overcome these shortcomings.

For this reason, the implementation in this notebook is intended purely for **educational purposes**. It demonstrates that, in principle, a wide range of neural network architectures—including simple feed-forward networks—can be trained as language models if they are given appropriately structured input-output pairs. By experimenting with this minimal setup, you will gain intuition for how context windows, embeddings, and prediction objectives interact, and why more advanced architectures are necessary for scaling to real-world language understanding tasks.

Let's check it out!

### Setting up the Notebook

#### Make Required Imports

This notebook requires the import of different Python packages but also additional Python modules that are part of the repository. If a package is missing, use your preferred package manager (e.g., [conda](https://anaconda.org/anaconda/conda) or [pip](https://pypi.org/project/pip/)) to install it. If the code cell below runs with any errors, all required packages and modules have successfully been imported.

In [1]:
import os, sys
import numpy as np
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset

from transformers import AutoTokenizer

from src.utils.compute.gpu import *
from src.utils.data.files import *

#### Download Required Data

Some code examples in this notebook use data that first need to be downloaded by running the code cell below. If this code cell throws any error, please check the configuration file `config.yaml` if the URL for downloading datasets is up to date and matches the one on Github. If not, simply download or pull the latest version from Github.

In [2]:
movie_reviews_zip, target_folder = download_dataset("text/corpora/reviews/movie-reviews-imdb.zip")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50.6M/50.6M [00:02<00:00, 19.8MiB/s]


We also need to decompress the archive file.

In [3]:
movie_reviews = decompress_file(movie_reviews_zip, target_path=target_folder)

print(movie_reviews)

['data/datasets/text/corpora/reviews/movie-reviews-imdb.txt']


#### Checking & Setting Computing Device

PyTorch allows to train neural networks on supported GPU to significantly speed up the training process. If you have a support GPU, feel free to utilize it. However, for this notebook it's certainly not needed as our dataset is small and our network model is very simple. We provide an auxiliary method to automatically select the best device. It checks if a supported GPU is available and if so, uses it as the preferred device.

In [4]:
# Select preferred device (GPU, if available; CPU otherwise); you can enfore the use of the CPU
DEVICE = select_device(force_cpu=False)

print("Available device: {}".format(DEVICE))

Available device: cuda:0


#### Preliminaries

Before checking out this notebook, please consider the following:

* This notebook is for education and not to build a state-of-the-art LLM. Most importantly, MLPs are not sequence models and therefore do not support inputs of variable lengths. And also, the dataset we use here is very small; it also stems from a single domain: movie reviews. As such, do not expect anything near human-like responses from the trained model, particularly in demo mode (see below).

* While not strictly required, we recommend the presence of a GPU to speed up the training. However, any more modern consumer GPU supported by the PyTorch library should be fine. Even for the full training mode, the default parameters are chosen that the training will not require more than 10 GB of VRAM; with 16GB slowly becoming the standard even for consumer GPUs.

* You can run the notebook in demo mode by using `mode = "demo"` in the code cell below. In demo mode, we only use 10,000 out of the 100,000 movie reviews. We recommend first using the demo mode to see how long the training of the model for individual epochs will require.

In [5]:
mode = "demo"
#mode = "full"

---

## Dataset Preparation

The [**ACL IMDB (Large Movie Review) dataset**](https://ai.stanford.edu/~amaas/data/sentiment/) is a popular benchmark dataset for sentiment analysis, introduced by Andrew Maas et al. (2011). It contains a total of 100,000 movie reviews collected from IMDb, with 50,000 reviews being labeled as either *positive* or *negative* and evenly split into 25,000 reviews for training and 25,000 for testing. For training Word2Vec word embeddings we do not need the sentiment labels. We therefore already preprocessed the original dataset such that all 100,000 reviews are in a single file, with 1 line = 1 review. This preprocessing included the removal of any HTML tags and line breaks.

For the training of our model, we make the common assumption that the training data is structured as continuous streams of documents, with each movie review representing a document. Document streams refer to the way text data is fed into the model during training: not as isolated documents, but as a continuous stream of text. Instead of treating each document as an independent unit, the training corpus is viewed as a long sequence of tokens coming from many documents concatenated together. This approach allows the model to learn language patterns and long-range dependencies across boundaries that would otherwise be artificially imposed by document splits. The training process still ensures that context resets appropriately between documents (e.g., by inserting special end-of-document tokens `[EOS]`), but from the model's perspective, the data flows in a seamless stream. Thus, assuming $\text{doc}_{i}$ is a list of tokens represents the $i$-th documents, our document stream has the following format:

$$\large
\text{doc}_{1} + \text{[EOS]} + \text{doc}_{2} + \text{[EOS]} + \text{doc}_{3} + \text{[EOS]} + \dots
$$

In practice, our training data may consist of many such document streams so that each stream can fit into memory. However, here we work with a small example data for educational purposes. This includes that the stream of tokens can be represented by an array that fits completely into memory, therefore avoiding any more sophisticated strategies requiring the splitting of the dataset, etc.

Document streaming is essential for efficient batching and scaling when training on massive corpora. LLMs are typically trained on trillions of tokens spread across billions of documents, and it's computationally impractical to load entire documents or reinitialize contexts for each. Instead, data pipelines tokenize all documents in advance, concatenate them, and then dynamically sample fixed-length chunks (e.g., 1024 tokens) to feed into the model. This "streaming" setup keeps GPUs fully utilized and allows distributed training systems to continuously stream data without needing to restart or reshuffle individual documents frequently.

### Load Reviews from File

In the setup section of the notebook, we already downloaded the file containing all 100,000 movie reviews. In the following code cell, simply counts the number of reviews by containing the number of each line in the file, just to check if the dataset is complete. Note that we have to write `movie_reviews[0]` since `movie_reviews` is a list of files &mdash; it just so happens that the list contains only one file.

In [6]:
total_reviews = sum(1 for _ in open(movie_reviews[0]))

print(f"Total number of reviews (1 review per line): {total_reviews}")

Total number of reviews (1 review per line): 100000


Although we have a total 100,000 reviews (each containing multiple sentences), we consider only 10,000 reviews in demo mode to speed up the training. However, you can edit the code cell below to increase or decrease the number of considered reviews. For a first run, we recommend sticking to 10,000 reviews to execute and understand the code.

In [7]:
if mode == "demo":
    num_considered_reviews = 10_000
else:
    num_considered_reviews = 100_000

num_reviews = min(total_reviews, num_considered_reviews)

print(f"Number of reviews used for training dataset: {num_reviews}")

Number of reviews used for training dataset: 10000


### Tokenize & Generate Token Stream

**Subword tokenization** such as Byte Pair Encoding (BPE), SentencePiece, or WordPiece became the standard for modern language models. Subword vocabularies strike a balance between characters and words: they are small enough to be efficient, yet expressive enough to represent any word by composing smaller units. This eliminates the `<unk>` problem entirely, improves handling of rare and morphologically rich words, and reduces vocabulary size dramatically, which helps models train more efficiently.

This is why we will be using subword tokenization in this notebook by means of a pretrained tokenizer. More specifically, we are using GPT-2 tokenizer to handle both the tokenization and indexing for us. The GPT-2 tokenizer is a subword tokenizer based on the **Byte-Pair Encoding (BPE)** algorithm. In nutshell, BPE (and similar approaches) learns how to tokenize words from data. Instead of splitting text strictly into full words or individual characters, BPE breaks words into frequently occurring subword units. For example, *"unhappiness*" might become *"un"*, *"happi"*, and *"ness"*. This enables the model to represent common words as single tokens while decomposing rare words into smaller, reusable pieces, preventing the out-of-vocabulary (OOV) problem that word-level tokenizers suffer from. Because subword tokens capture meaningful morphological patterns (prefixes, suffixes, roots), models can infer the meaning of new or compound words based on familiar subparts. This reduces the total number of parameters needed for embeddings and improves the model's ability to generalize to unseen text. You can learn all about BPE in a separate notebook

To load the pretrained GPT-2 tokenizer, we can use the `AutoTokenizer` class Hugging Face Transformers library. This class is a high-level interface that automatically selects the appropriate tokenizer for a given pretrained model, abstracting away the need to know the specific tokenizer class. The `from_pretrained` method is used to load a tokenizer that has already been trained on a specific model's vocabulary and tokenization rules, either from the Hugging Face model hub or a local path. This method ensures that the tokenizer is configured consistently with the pretrained model, including token-to-ID mappings, special tokens, and any subword tokenization schemes (e.g., BPE), making it ready for encoding text into token IDs suitable for model training or inference. The code cell below used the class and method to load the GPT-2 tokenizer.

In [8]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

Recall that for our document stream, we need to indicate when one movie review ends and another review starts using some `[EOS]` (end-of-sequence) token. However, we cannot simply define our own unique token but must use a token that is known to the tokenizer, i.e., the token is part of the existing vocabulary. Most tokenizers include a small set of special tokens to indicate the end of a sequence, the beginning of a sequence, padding tokens, masked tokens, etc. &mdash; all depending on the data and learning task.

We can check the `special_tokens_map` of the GPT-2 tokenizer which special tokens it supports:

In [9]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

We can see that the GPT-2 tokenizer recognizes only one special token: `<|endoftext|>`. Since GPT-2 was also trained on a document stream, we only need a single token acting as a separator between documents, which could be either the `[EOS]` or `[BOS]` token. The GPT-2 tokenizer also does not require a dedicated `[UNK]` (unknown) token, since BPE just tokenizes unknown words into known subwords or even just characters, if needed.

Let's define this `<|endoftext|>` token as a constant for creating our document stream. Using the tokenizer instance, we can also encode this special token to get its unique index. We will need this index later when generating text using our RNN-based language model.

In [10]:
EOS_TOKEN = "<|endoftext|>"
EOS_TOKEN_INDEX = tokenizer.encode(EOS_TOKEN)[0]

With the tokenizer, we can now go through all movie reviews (or the maximum number of reviews specified) and tokenize them; see the code cell below. Notice that the preprocess each review before tokenizing by removing any newline characters, converting all words to lowercase, and adding the special `[EOS]` token at the end. 

Lowercasing all words can be very useful when working with smaller datasets because it significantly reduces the size of the vocabulary the model needs to learn. In a small corpus, many words appear only a handful of times, and treating uppercase and lowercase forms as separate tokens (e.g., *"Movie"* vs. *"movie"*) further fragments the data. By converting everything to lowercase, the model encounters each word more frequently, allowing it to learn better embeddings and stronger statistical patterns from limited examples. This makes training more stable and reduces the risk of overfitting to infrequent capitalized variants.

Additionally, the goal in many educational or exploratory projects is to understand the mechanics of training sequence models and not to capture subtle linguistic nuances such as proper noun capitalization. Lowercasing simplifies preprocessing, reduces noise, and helps the model focus on learning the core structure of the language rather than expending capacity on orthographic variations. For small-scale experiments, this trade-off is highly beneficial: you get cleaner data, faster training, and more reliable results without sacrificing the insights the project aims to teach.

In [11]:
tokens = []

with open(movie_reviews[0]) as file:
    for idx, review in enumerate(tqdm(file, total=num_reviews, leave=False)):
        if idx >= num_reviews:
            break
        tokens.extend(tokenizer.encode(f"{review.strip().lower()} {EOS_TOKEN}", truncation=True, max_length=sys.maxsize))

print(f"Total number of tokens: {len(tokens)}")

Total number of tokens: 2922984


### Create `Dataset` and `DataLoader`

PyTorch's `Dataset` class provides a clean and modular way to represent and access your data. It allows you to define how individual samples are loaded, whether they come from tokenized text, images on disk, or preprocessed tensors. This abstraction keeps the data logic separated from the model and training loop, making your code easier to maintain and allowing you to swap in different datasets without changing the rest of your pipeline. The `DataLoader` builds on top of a `Dataset` to efficiently feed data to the model during training. It handles batching, shuffling, and parallel data loading through worker processes, which significantly speeds up training and keeps the GPU fully utilized. Together, `Dataset` and `DataLoader` provide a flexible and scalable framework for handling data in PyTorch, allowing you to focus on model design and experimentation rather than manually managing data batching or iteration logic.

### Batching Considerations

Right now, we have with `tokens` a very long list of token IDs representing our document stream. However, cannot directly pass this list as training data to our MLP later due to two main considerations

* **Fixed input size:** MLPs inherently assume inputs of a fixed dimensionality, since each layer is defined by weight matrices with predetermined shapes. As a result, when dealing with sequential data such as text, the variable-length nature of sequences becomes incompatible with this requirement. To bridge this gap, sequences must first be segmented or padded into fixed-length chunks before being fed into the model. This preprocessing step ensures that every input conforms to the expected size.

* **Batch processing:** In addition to fixed input lengths, MLPs are typically designed to process data in batches to improve computational efficiency. By stacking multiple fixed-length sequences into a single tensor, modern hardware like GPUs can exploit parallelism, performing the same operations on many examples simultaneously rather than one at a time. This batching significantly speeds up training and makes better use of memory bandwidth, but it further reinforces the need for all inputs in a batch to share the same shape—hence the requirement to split or pad sequences to a uniform fixed length beforehand.

A common approach to handle long sequences of token IDs is to reshape the single list into multiple parallel streams, where the number of streams corresponds to the chosen `batch_size`. Instead of processing one long sequence end-to-end, the token list is split into equally sized chunks and arranged column-wise, so that each stream represents a continuous subsequence of the original data. This allows the model to process several subsequences in parallel, improving efficiency while preserving local sequential structure within each stream. 

To illustrate this idea, let's assume we have the following list of tokens IDs &mdash; for simplicity the token IDs go from $1$ to $32$:

```
tokens = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, ..., 30, 31, 32]
```

Let's further assume that we want to train our model with a batch size of $3$, i.e., `batch_size = 3`. Since our current number of tokens (i.e., `32`) is not divisible by $3$, we drop the last two tokens; keep in mind that when working with document streams containing millions and billions of tokens, this will not have any negative effect. With a total of now $30$ token IDs, we can split the list into three streams representing continuous subsequences of the original IDs:

```
Stream 1: [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]
Stream 2: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Stream 3: [21, 22, 23, 24, 25, 26, 27, 28, 29, 20]
```

Lastly, we have to consider the fixed input size of MLP by moving a sliding window of a fixed size across all streams. For example, let's consider a context of $5$ tokens &mdash; that is, we want to train a model that predicts the next token based on the previous $5$ tokens. This means that our first input batch looks as follows:

```
[ 1,  2,  3,  4,  5]
[11, 12, 13, 14, 15]
[21, 22, 23, 24, 25]
```

The second batch then has the following form:

```
[ 2,  3,  4,  5,  6]
[12, 13, 14, 15, 16]
[22, 23, 24, 25, 26]
```

and so on until we reach the end of the streams.

The target for each input sequence naturally derives from the next word predict task. For our example, for each sequence of $5$ token IDs, we want to predict the token ID that directly follows those $5$ IDs. Thus, the complete first batch with both the inputs and the targets looks as follows:

```
[ 1,  2,  3,  4,  5], [ 6]
[11, 12, 13, 14, 15], [16]
[21, 22, 23, 24, 25], [26]
```

### Implementing Custom `DataSet` Class

Now that we know in which format we have to bring our training data, we can implement a custom `DataSet` to achieve this. More specifically, create a class that inherits from the [IterableDataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.IterableDataset) class. This dataset class is designed for scenarios where data must be generated or streamed sequentially rather than randomly accessed. Unlike the standard `Dataset` class, which assumes that individual samples can be indexed (`__getitem__()`), an `IterableDataset` only requires the implementation of an `__iter__()` method that yields data samples one at a time. This makes it ideal for use cases such as reading data from large text files, streaming logs, or generating data on the fly &mdash; situations where random access is impossible or inefficient.

Because `IterableDataset` produces samples in a fixed order, it gives the user precise control over the data flow, which is especially useful for tasks like language modeling with long text streams, reinforcement learning, or online learning. However, this sequential nature also means that certain DataLoader features (e.g., shuffling or automatic batching) are limited or behave differently. This includes that we have to implement batching ourselves &mdash; but we know how to do it (see the figure above). Despite these constraints, `IterableDataset` is a powerful tool when working with continuous or very large datasets that don't fit neatly into memory or require sequential processing.

The class `StreamDataset` in the code cell below implements the approach. When passing the initial array of `tokens`, the `batch_size`, and the sequence length `context_size` of all sequences within a batch, the constructor performs the following three main steps:

**(1) Compute the number of usable tokens:** This step refers to ignoring the last tokens in the inputs token list to create `batch_size` streams of equal length. We can compute this number by first computing the number of full batches by dividing the total number of tokens by the batch size using an integer division (without remainder). The number of usable tokens is the number of batches, and then multiplied again by the number of batches.

**(2) Create substreams:** We can create all `batch_size` substreams by reshaping the original 1-dim array of tokens into a 2-dim array of shape `(batch_size, stream_len)` where `stream_len` is the number of tokens in each substream &mdash; this number is, of course, the number of usable tokens divided by the batch size (i.e., the number of streams). Having all streams in a single array makes creating the final batches very easy.

**(3) Compute total number of batches:** The number of batches is naturally directly related to the length of each stream since each batch derives from the fixed-sized sliding window we move across the streams. However, we have to stop the sliding window $t$ token ID before the end of the stream since the last token ID in each stream will form the target for the last batch. The number of batches is therefore simply the length of the streams `stream_len` minus the length of each input sequence `context_size`.

In the `__iter__()` method simply iterate overall valid batches numbers (i.e., their index from `0` to `n_batches-1`) to generate all input sequences and target sequences by slicing the 2-dim array that holds all substream.

In [12]:
class StreamDataset(IterableDataset):

    def __init__(self, tokens: torch.LongTensor, batch_size: int, context_size: int):
        super().__init__()
        self.tokens = tokens
        self.batch_size = batch_size
        self.context_size = context_size
        # (1) Compute number of usable tokens
        total_tokens = tokens.size(0)
        usable = (total_tokens // batch_size) * batch_size
        self.tokens = tokens[:usable].contiguous()
        self.stream_len = usable // batch_size  # length of each stream
        # (2) Create substreams: reshape into (batch_size, stream_len)
        self.streams = self.tokens.view(batch_size, self.stream_len)
        # (3) Compute total number of batches: ignore the last batch, even if it is full
        self.n_batches = (self.stream_len - context_size)

    def __iter__(self):
        s = self.context_size
        # Generate next pair of inputs and targets
        for step in range(self.n_batches):
            inputs = self.streams[:,(step):(step+s)]
            targets = self.streams[:,(step+s)]
            yield inputs, targets

To show how it works, we can replicate the example as visualized in the figure about. In the code cell below, we first create a 1-dim array with the numbers from $1$ to $60$. We use this array to create a `StreamDataset` instance, assuming a batch size of $3$ and a sequence length per batch of $5$.

In [13]:
demo_tokens = torch.arange(1, 32)

demo_dataset = StreamDataset(demo_tokens, batch_size=3, context_size=5)

To easily iterate over all batches, we first create a `DataLoader` instance using our demo dataset as input. Since the batching is handled by the `StreamDataset` class, and we can use shuffling &mdash; otherwise we would break the continuation between batches, we basically only pass the dataset instance to the data loader class. Lastly, we can use the data loader to easily iterate over all batches; and print them shown below.

In [14]:
demo_loader = DataLoader(demo_dataset, batch_size=None)

for step, (inputs, targets) in enumerate(demo_loader):
    print(f"========== Batch {step+1} ==========")
    print(f"Inputs:\n{inputs}")
    print(f"Target:\n{targets}\n")

========== Batch 1 ==========
Inputs:
tensor([[ 1,  2,  3,  4,  5],
        [11, 12, 13, 14, 15],
        [21, 22, 23, 24, 25]])
Target:
tensor([ 6, 16, 26])

========== Batch 2 ==========
Inputs:
tensor([[ 2,  3,  4,  5,  6],
        [12, 13, 14, 15, 16],
        [22, 23, 24, 25, 26]])
Target:
tensor([ 7, 17, 27])

========== Batch 3 ==========
Inputs:
tensor([[ 3,  4,  5,  6,  7],
        [13, 14, 15, 16, 17],
        [23, 24, 25, 26, 27]])
Target:
tensor([ 8, 18, 28])

========== Batch 4 ==========
Inputs:
tensor([[ 4,  5,  6,  7,  8],
        [14, 15, 16, 17, 18],
        [24, 25, 26, 27, 28]])
Target:
tensor([ 9, 19, 29])

========== Batch 5 ==========
Inputs:
tensor([[ 5,  6,  7,  8,  9],
        [15, 16, 17, 18, 19],
        [25, 26, 27, 28, 29]])
Target:
tensor([10, 20, 30])



Appreciate how the input sequences match the example batches in the figure above.

Of course, we do not want to train our model based on this demo dataset but on our movie reviews. So let's create a data loader using our initial document stream `tokens` with a batch size of $256$ and a context size of $10$. The chosen context size (i.e., the number of input tokens) directly limits how much prior information the model can use to predict the next word. A small context size makes the model rely only on very local patterns, which is computationally efficient but often leads to poorer predictions because longer-range dependencies are ignored. Increasing the context size can improve performance by giving the model more information about preceding tokens, but it comes at a cost: the input dimensionality grows linearly with the context length, which quickly increases the number of parameters and computational requirements. As a result, there is a practical trade-off between capturing richer context and maintaining a manageable, efficient model.

In [15]:
context_size = 10

dataset = StreamDataset(torch.LongTensor(tokens), batch_size=256, context_size=context_size)

loader = DataLoader(dataset, batch_size=None)

Keep in mind that creating the dataset and data loader was rather straightforward since the dataset is small enough to easily fit into memory. In contrast, large training datasets typically cannot fit into main memory all at once. Thus, when working with extensive text corpora (often gigabytes in size) loading everything upfront would exceed memory limits. Instead, the data must be streamed, read in chunks, or processed on the fly. This requires dataset implementations that can efficiently iterate through the data without holding it entirely in memory. Because of these constraints, additional considerations are needed when creating `Dataset` and `DataLoader` instances. For example, one may extend `IterableDataset` implementation to read from large text files line by line, implement custom buffering strategies, or perform on-demand tokenization. However, these considerations go beyond the scope of this notebook and are not needed here.

---

## Auxiliary Methods

For the model training and the very crude qualitative evaluation of the model (discussed later), we next define a few auxiliary models for a cleaner code but also support strategies such as checkpointing for training large models in practice.

### Training a Single Epoch

As the name suggests, the method `train_epoch()` trains a given model for one epoch. Apart from the model itself, the method also gets a data loader instance to iterate over all batches, as well as an optimizer to handle the update of the model weights after the gradients have been computed during backpropagation; the description is merely to annotate the progress bar.

In [16]:
def train_epoch(loader, model, optimizer, description):
    model.train()
    
    epoch_loss = 0.0
    device = next(model.parameters()).device
    
    for idx, (inputs, targets) in enumerate(tqdm(loader, desc=description, leave=False, total=dataset.n_batches)):
        # Move data to the same device as the model
        inputs = inputs.to(device)
        targets = targets.to(device)
        # Pass inputs to model to get the logits as outputs
        logits = model(inputs)  # logits: (batch_size, vocab)
        # Compute loss: flatten seq and batch dims for CrossEntropyLoss
        loss = criterion(logits, targets)
        # Perform PyTorch magic: backpropagation + parameter updates
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)  # optional grad clipping
        optimizer.step()

### Saving & Loading Checkpoints

Large models require a lot of training, even for a rather small model trained using a rather small dataset. A **checkpoint** in model training is a saved snapshot of the model's state at a specific point during training, typically after a certain number of steps or epochs. It usually includes the model's parameters (weights and biases), the optimizer state (to resume learning with the same momentum and learning rate adjustments), and sometimes metadata like the current epoch or training loss. This allows training to be paused and later resumed from that point without starting over, which is especially important for large models that take days or weeks to train.

While many libraries have built-in support for periodically saving checkpoints, in this notebook, we purposefully use only PyTorch and avoid libraries with a higher level of abstraction. However, saving a checkpoint is very straightforward. The method `save_checkpoint()` defined in the code cell below takes a model and optimizer instance, as well as the information about the current epoch and epoch loss. The method combines all required information to resume training into a single object and uses the `save()` method of PyTorch to save this object to a file.

In PyTorch, the `state_dict()` object is a Python dictionary that contains all the learnable parameters and persistent states of a model or optimizer. For models, it stores mappings from each layer's name to its corresponding tensor values (like weights and biases). For optimizers, it includes the current state of optimization variables such as momentum buffers and learning rate schedules. This dictionary enables easy saving, loading, and transferring of model and optimizer states, making it central to checkpointing and model deployment. By calling `torch.save(model.state_dict())`, you can preserve a model's learned parameters, and later restore them with `model.load_state_dict()`, ensuring the model continues exactly where it left off.

In [17]:
def save_checkpoint(model, optimizer, epoch, loss, path="checkpoint.pt"):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved at {path}")

Naturally, the counterpart to saving a checkpoint is loading a save checkpoint, as implemented by the method `load_checkpoint()` below. Notice that this method also takes in a model and optimizer instance. In other words, the method does not create a model or optimizer but sets the state of both instances as the states read from the checkpoint file. This of course only works if the model and the optimizer have the same "structure" as the model and optimizer used for training. For example, we cannot train a Transformer model with 4 layers and then load its state into a model with more or less layers.

Also, notice the `map_location` parameter of PyTorch's `load()` method. This parameter controls how tensors are remapped to devices (like CPU or GPU) when loading a saved model or checkpoint. This is useful when the model was trained on one device but needs to be loaded on another; for example, loading GPU-trained weights onto a CPU-only machine. By specifying `map_location='cpu'`, all tensors are loaded to the CPU regardless of where they were originally saved, while `map_location='cuda:0'` loads them to the first GPU. It can also take a custom function or dictionary to flexibly remap devices, ensuring model compatibility across different hardware setups and preventing errors caused by unavailable devices.

In [18]:
def load_checkpoint(model, optimizer, path="checkpoint.pt", device="cuda"):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    epoch = checkpoint["epoch"]
    loss = checkpoint["loss"]
    print(f"Checkpoint loaded (epoch {epoch}, loss {loss:.4f})")
    return epoch, loss

### Generate & Save Example Responses

When training any kind of model, we typically track the progress by measuring some form of validation loss over time. However, here we keep it very simple and omit the consideration of a separate validation dataset to compute a validation loss after each epoch. After all, the loss itself does not really tell us how well the model is performing. In contrast, we first define the method `generate_response()` that returns the response generated by a model for a given prompt. Notice that this method also needs the tokenizer to encode the prompt and decode the generated response.

In [19]:
def generate_response(prompt, tokenizer, model, eos_token=None, max_new_tokens=50):
    # Identify the device where the model is location
    device = next(model.parameters()).device
    # Encode prompt using the tokenizer
    prompt_indices = torch.LongTensor(tokenizer.encode(prompt))
    # Use model to generate the next tokens
    generated_indices = model.generate(prompt_indices, eos_token, device=device)
    # Decode and return sequence of indices into human-readable tokens
    return tokenizer.decode(generated_indices)

Similar to saving checkpoints when training the model for a long time, we might also want to save the generated responses for some prompts, e.g., after each epoch. This allows us to later check the generated files, how the quality of the generated responses have changed over time. The method `generate_example_responses()` implements this idea using three simple example prompts related to movie reviews; of course, we can edit the prompts or additional ones. All generated responses are stored in a file given the specified file name.

In [20]:
def generate_example_responses(tokenizer, model, eos_token, path="example-responses.txt"):
    prompts = ["the best part of the movie was", "my favorite scene in the movie", "the script and the direction"]
    with open(path, "w") as file:
        for prompt in prompts:
            response = generate_response(prompt, tokenizer, model, eos_token)
            file.write(f"{response}\n\n")

In the full training mode, we save checkpoints and example responses to disk. Using the code cell below, you can create a folder into which the checkpoint and output files are stored. You can customize the path to fit your local setup.

In [21]:
folder = create_folder("data/generated/models/mlp-lm/")

print(folder)

data/generated/models/mlp-lm/


---

## Creating & Training the Model

### Model Definition

In this notebook, we will be implementing an MLP with $2$ hidden layers, say, with $512$ neurons each. The size of the input layer depends on two parameters:

* `context_size` &mdash; the number of tokens (i.e., the length of the sequences) to be used to predict the next token (e.g., $5$)

* `embed_size` &mdash; the size of the embedding vectors for each token (e.g., $100$)

Thus, the size of the input layer is `context_size * embed_size`. Naturally, the size of the output layer derives from the size of the vocabulary since the output needs to be the probability for each token given the input tokens. Since we are using the pretrained GPT-2 tokenizer with a vocabulary of $50,257$ unique tokens, the size of the output layer is $50,257$. The figure below illustrates this idea assuming a context size of $5$. Keep in mind that all layers are fully connected, but not all connections are shown to ease presentation. The example also considers full words instead of subwords; again, this is merely to make the figure easier to understand.

<img src="images/illustrations/mlp/mlp-language-model-example-setup.png" style="margin:auto;max-width:850px;width:100%" alt="2-Layer MLP">

The class `MlpLM` implements this setup. Note that, compared to the figure above, it only adds an `nn.Embedding` layer to map the token IDs to their respective token indices. Notice how the first fully connected hidden layer `fc1` expects an input of size `context_size * embed_size` as previously motivated. The `forward()` method is straightforward, pushing the input through all the layers to get the final output logits; note that this implementation is using the ReLU activation function.

Most of the code is actually the `generate()` function to generate an output sequence for a given sequence of input token IDs. Since we have to ensure a fixed input size for the model during each iteration, the implementation of the method has to handle to corner cases:

* **Padding (optional)**: In case the initial sequence of tokens is shorter than the context size, we have to pad the sequence to the context size. We do this by prepending the "enough" EOS token ID to the current sequence of token IDS

* **Cutting:** If the initial sequence of token IDs or enough new token IDs have been predicted, the current sequence of all token IDs so far will be longer than the context size. In this case, we only considered the last `context_size` token IDs for the next iteration and thus prediction for the next token.

Apart from ensuring the correct input size, the `generate()` method also performs *top-k* sampling &mdash; that is, instead of always picking the token with the highest probability (i.e., *greedy decoding*), the method considers the top-k tokens and randomly picks a token from this set proportional to their probabilities. This enables a level of creativity, as greedy decoding would always generate the exact same sequence for the same input.

In [22]:
class MlpLM(nn.Module):
    def __init__(self, vocab_size, context_size, embed_size, hidden_size):
        super().__init__()
        self.context_size = context_size
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.fc1 = nn.Linear(context_size * embed_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embeds = self.embeddings(x)              # (batch, context_size, embed_size)
        x = embeds.view(embeds.size(0), -1)      # flatten: (batch, context_size*embed_size)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        logits = self.out(x)
        return logits

    @torch.no_grad()
    def generate(self, seed_indices, eos_token_id=None, max_len=50, temperature=1.0, top_k=10, device='cpu'):
        # Initialize list of generation tokens with seed tokens
        generated = seed_indices.tolist()
        # Generate remaining tokens step by step
        for _ in range(max_len):
            # Take last context_size tokens
            context = generated[-self.context_size:]
            # Pad with EOS token id if needed (for short initial input)
            if len(context) < self.context_size:
                context = [eos_token_id] * (self.context_size - len(context)) + context
            # Construct input batch (batch_size=1) and move to correct device
            context_tensor = torch.tensor(context, dtype=torch.long, device=device)
            context_tensor = context_tensor.unsqueeze(0)  # (1, context_size)
            # Perform forward pass to get logits
            logits = self.forward(context_tensor)  # (1, vocab_size)            
            # Apply temperature to logits
            logits = logits / temperature
            # Top-k filtering (if specified)
            if top_k is not None and top_k < logits.size(-1):
                topk_vals, topk_idx = torch.topk(logits, top_k)
                probs = F.softmax(topk_vals, dim=-1)
                next_token = topk_idx.gather(1, torch.multinomial(probs, num_samples=1))
            else:
                # fallback to full softmax sampling
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            # Stop generating tokens if the next token is the EOS token
            if eos_token_id is not None:
                if next_token.item() == eos_token_id:
                    break
            # Add new token to the final list of tokens (i.e., token indices)
            generated.append(next_token.item())
        # Return final list of all tokens (seed tokens + generation tokens)
        return generated

The output of the `generate()` method is the sequence of input and generated token IDs. This sequence we can then give to the tokenizer class to decode the token IDs to proper words; cf. the auxiliary method `generate_response()` above.

### Model Training

To train the model, we first need to instantiate the model itself, as well as the optimizer and the criterion (i.e., the loss function); just run the code cell below. For this example, we use the **Adam** optimizer, one of the most popular optimizers to train neural networks. Regarding the criterion, keep in mind that the `nn.CrossEntropyLoss` works directly on logits because it internally applies `log_softmax` to convert them into log-probabilities before computing the negative log-likelihood. This design is both numerically stable and efficient: applying `softmax` and then taking a log would risk floating-point underflow, but combining them into a single `log_softmax` step avoids these issues. As a result, you can feed raw, unnormalized model outputs (logits) directly into `nn.CrossEntropyLoss`, and it will safely and correctly compute the cross-entropy without requiring you to apply softmax yourself.

**Important:** Note that we have to use the same context size we used to build the data loader; the embedding size and hidden size we can choose more freely in the code cell below. So in the case you also want to play with the context size, you need to go back and change the parameters when creating the data loader.

In [23]:
embed_size, hidden_size = 100, 512

model = MlpLM(tokenizer.vocab_size, context_size, embed_size, hidden_size).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(model)

MlpLM(
  (embeddings): Embedding(50257, 100)
  (fc1): Linear(in_features=1000, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (out): Linear(in_features=512, out_features=50257, bias=True)
)


Before we actually start with the training, let's first see how the model behaves with its randomly initialized weights. For this, we can use the `generate_response()` method and a simple prompt to let the model generate the next best tokens; see the code cell below.

In [24]:
prompt = "the best part of the movie was"

generate_response(prompt, tokenizer, model, EOS_TOKEN_INDEX)

'the best part of the movie was upfrontrecogn € Tara sucks harbor commissioner thee Mets inland Farms unatt clandestine Tara Lopez953isner monetary witnessingScience aversci Sendplant Ara brunch Tara manufacture awaitedrenchesish Tara Marvel awaited Notre illuminate 133 entranceitating nude>>Moreover revival reapp Dough milk codedTags laureate010'

As expected, beyond the seed words of the prompt, the output looks like a random garbled mess. However, we can use this output to compare it with one after even the first epoch.

Finally, we are ready to train the model. In the code cell below, by default, we train the model for $5$ epochs. After each epoch we either print a single generated response based on the previous example prompt (demo mode) or generate multiple example responses and save them to a file (full training mode). In the full training mode, we also save a checkpoint after each epoch.

**Important:** While the code below saves a checkpoint after each epoch in full training mode, it does not directly resume training after a problem by automatically loading the last checkpoint; notice that the method `load_checkpoint()` is actually nowhere used in the notebook. This is to keep the code as simple and clean as possible. Also, even in full training mode &mdash; and assuming a decent consumer GPU &mdash; the training time is measured in very few hours instead of days.

Also word of "warning": After only $5$ epochs, even when using all $100,000$ reviews in the full training mode, you are unlikely to see great results in terms of human-like responses. Properly training a neural network-based language model simply requires a lot of time and computational resources. We also work only with a small and domain-specific dataset that likely contains not (very) well-formed sentences. We also do not perform any kind of hyperparameter tuning in this notebook to find the best values for the context size, embedding size, hidden size, number of layers, learning rate, etc. However, you should still be able to see how the generated responses improve over time, particularly compared to the example generated without any training (see above).

In [25]:
num_epochs = 5

for epoch in range(num_epochs):
    description = f"Epoch {epoch+1}/{num_epochs}"
    epoch_loss = train_epoch(loader, model, optimizer, description)
    # Generate some responses to track progress
    if mode == "demo":
        print(generate_response(prompt, tokenizer, model, eos_token=EOS_TOKEN_INDEX))
    else:
        save_checkpoint(model, optimizer, epoch+1, epoch_loss, path=f"{folder}checkpoint-{epoch+1}.pt")
        generate_example_responses(tokenizer, model, eos_token=EOS_TOKEN_INDEX, path=f"{folder}example-responses-{epoch+1}.txt")

print(f"Done training {num_epochs} epochs.")

the best part of the movie was, it was so much of all the time, and there is a bad one of the characters. it is a few and the movie.  that. in that is not a good movie. there were so bad i can't say, i don


the best part of the movie was in a film that are about it with a few minutes, the whole film. the film has been so a bad movie, the plot line, but this is the most part, and it was just so bad. i was looking for the film.


the best part of the movie was a lot of movie, and then they were all. in the end of a lot and i'm not sure if you have been seen and you can see the same movie, but the film was a bit of the story. 


the best part of the movie was not worth it, and it is one or two are a little to have to be the most. ursas. there are some other reviews in this, it's all to be the case. but i can't get a good idea in the


the best part of the movie was very funny, but i have to see a lot of film. it's just not the most, and that's it's a good thing, and the acting is so boring and a lot of a lot of good thing that has been more to the
Done training 5 epochs.


In demo mode and after only a handful of epochs, you should be able to see the responses slowly become better, with short phrases becoming more and more coherent. However, getting the model to generate fluent and coherent response across the whole sequence requires a substantial amount of training and a sufficiently large and tuned model.

### Discussion

In principle, we could train our model for more epochs, using more and diverse data, or perform hyperparameter tuning to improve the results. However, keep in mind that the core issue is that an MLP has **no built-in notion of sequence or memory**. It treats its input as a fixed-size vector, meaning the model can only consider a limited context window (e.g., the last $n$ words/tokens). Natural language, however, often depends on relationships that span long distances &mdash; such as maintaining topic consistency, resolving references, or preserving grammatical structure across sentences. Capturing these long-range dependencies with an MLP would require either an impractically large input size or inefficient workarounds, neither of which scales well.

In addition, MLPs do not share parameters across positions in a sequence. Each input position is effectively treated independently, which prevents the model from learning general patterns like "a verb often follows a subject" in a position-invariant way. Architectures such as recurrent networks or transformers explicitly address this by **reusing parameters across time steps** or modeling **interactions between all tokens**, making them far more suitable for language generation tasks. Another practical limitation is efficiency. For large vocabularies and longer contexts, the number of parameters in an MLP grows quickly, making training and inference expensive without delivering competitive performance. This is one reason why modern language models rely on architectures designed specifically for sequential data.

That being said, MLP-based language models are far from useless. They can be quite effective in tasks where the goal is not to generate long, coherent text, but rather to **score or rank alternatives given a limited context**. For example, in **spell checking**, the model can evaluate which candidate word best fits a short surrounding context. Similarly, in **speech recognition**, an MLP can help determine which word sequence is more likely among several hypotheses produced by an acoustic model. In these cases, the limited context and lack of sequential structure are less problematic, and the simplicity of an MLP can even be an advantage.

---

## Summary

This notebook provided a complete, end-to-end tutorial on training a simple language model using a Multilayer Perceptron (MLP). Starting from a small toy dataset, it walked through every step of the pipeline: tokenization, building a vocabulary, converting text into sequences of token IDs, and constructing fixed-length context–target pairs suitable for supervised learning. Special attention was given to reshaping sequential data into a format compatible with MLPs, including splitting sequences into fixed-size contexts and organizing them into batches for efficient training.

The core of the notebook focused on implementing an MLP-based language model in PyTorch. The model used an embedding layer to represent tokens, followed by two fully connected hidden layers and the output layer to predict the next word given a fixed context window. Along the way, the notebook demonstrated how design choices such as context size, embedding dimension, and hidden layer width affect both the model’s capacity and computational cost. Training procedures, loss computation, and basic evaluation were also covered, giving a clear picture of how such a model learns from data.

Importantly, the notebook emphasized the limitations of MLPs for language modeling. Because the model relies on a fixed context size, it cannot naturally capture long-range dependencies in text, and scaling it quickly becomes inefficient. These constraints highlight why more advanced architectures such as recurrent or attention-based models are typically preferred in practice. The main takeaway is conceptual: the next-word prediction objective that underlies language modeling is not tied to a specific architecture. Even a simple MLP can be used to formulate and learn this task, as long as the data is prepared appropriately. The notebook therefore serves as an educational tool to demystify language modeling, showing that while modern models are more sophisticated, the fundamental learning problem remains the same.